# 2. Data Preprocessing & Feature Engineering

In this section, we prepare our raw dataset for machine learning models. Instead of passing the entire raw history directly into sequential models, we will build a structured tabular dataset. We will aggregate the customer's historical shopping behavior into statistical features, extract the final quote before purchase, handle missing values, and encode categorical variables.

---

### 2. İlkin Məlumat Emalı və Yeni Xüsusiyyətlərin Yaradılması (Feature Engineering)

Bu bölmədə biz xam datasetimizi maşın öyrənməsi modelləri üçün hazır vəziyyətə gətiririk. Bütün alış-veriş tarixçəsini birbaşa mürəkkəb sıralı modellərə vermək əvəzinə, strukturlaşdırılmış cədvəl datası quracağıq. Müştərinin keçmiş shopping davranışlarını statistik xüsusiyyətlərə çevirəcək, satınalmadan əvvəlki son təklif sətirini götürəcək, çatışmayan dəyərləri dolduracaq və kateqorial dəyişənləri kodlaşdıracağıq.

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("train.csv")

# Separate shopping stages (record_type = 0) and final purchase stages (record_type = 1)
shopping_df = df[df["record_type"] == 0].copy()
purchase_df = df[df["record_type"] == 1].copy()

print("Shopping records shape:", shopping_df.shape)
print("Purchase records shape:", purchase_df.shape)

Shopping records shape: (568240, 25)
Purchase records shape: (97009, 25)


### Observation

We separated the dataset into two components: `shopping_df` containing the interaction history and `purchase_df` containing the final choices. This separation prevents **data leakage**, ensuring that features are only derived from the shopping process, and target labels are derived strictly from the final purchase event.

### Şərh / Müşahidə

Dataseti iki hissəyə ayırdıq: müştərinin keçmiş addımlarını ehtiva edən `shopping_df` və yekun qərarları ehtiva edən `purchase_df`. Bu ayrılma **data leakage (məlumat sızması)** probleminin qarşısını alır; beləliklə, xüsusiyyətlər yalnız alış-veriş prosesindən, hədəf class-lar (targets) isə ciddi şəkildə yalnız satınalma anından götürülür.

In [3]:
# Grouping by customer_ID to extract historical statistics from shopping experiences
history_features = shopping_df.groupby("customer_ID").agg(
    total_shopping_pts=("shopping_pt", "max"),
    cost_mean=("cost", "mean"),
    cost_min=("cost", "min"),
    cost_max=("cost", "max"),
    cost_std=("cost", "std"),
    unique_cost_count=("cost", "nunique")
).reset_index()

# Fill NaN in cost_std (occurs if a customer has very few points, though min points is 3, std is safe)
history_features["cost_std"] = history_features["cost_std"].fillna(0)

# Calculate cost difference between first and last shopping point
first_quotes = shopping_df.sort_values(["customer_ID", "shopping_pt"]).groupby("customer_ID").first()["cost"].reset_index().rename(columns={"cost": "first_cost"})
last_quotes = shopping_df.sort_values(["customer_ID", "shopping_pt"]).groupby("customer_ID").last()["cost"].reset_index().rename(columns={"cost": "last_cost"})

# Merge cost diff features
cost_diff_df = pd.merge(first_quotes, last_quotes, on="customer_ID")
cost_diff_df["cost_change_amount"] = cost_diff_df["last_cost"] - cost_diff_df["first_cost"]

history_features = pd.merge(history_features, cost_diff_df[["customer_ID", "cost_change_amount"]], on="customer_ID")
history_features.head()

,customer_ID,total_shopping_pts,cost_mean,cost_min,cost_max,cost_std,unique_cost_count,cost_change_amount
0,10000000,8,633.375000,630,638,3.961872,3,5
1,10000005,5,745.200000,730,755,13.423859,3,-24
2,10000007,7,609.285714,605,618,5.376315,4,-13
3,10000013,3,625.333333,620,629,4.725816,3,-2
4,10000014,5,599.600000,585,604,8.173127,3,-1


### Observation

We successfully engineered historical features for each unique customer. Instead of losing the time-series data, we condensed it into meaningful behavioral metrics such as price volatility (`cost_std`), the number of times the quote price changed (`unique_cost_count`), and whether the price went up or down from the first quote to the last (`cost_change_amount`).

### Şərh / Müşahidə

Hər bir unikal müştəri üçün tarixçəyə əsaslanan yeni xüsusiyyətləri uğurla yaratdıq. Zaman ardıcıllığı məlumatını tamamilə itirmək əvəzinə, onu qiymət dəyişkənliyi (`cost_std`), təklif olunan qiymətin neçə dəfə dəyişdiyi (`unique_cost_count`) və qiymətin ilk təklifdən sona doğru nə qədər artıb-azaldığı (`cost_change_amount`) kimi mənalı davranış göstəricilərinə çevirdik.

In [4]:
# Identify and extract the last shopping point row for each customer
last_shopping_point = shopping_df.sort_values(["customer_ID", "shopping_pt"]).groupby("customer_ID").last().reset_index()

# Drop columns that are no longer needed or will be replaced
last_shopping_point = last_shopping_point.drop(columns=["record_type", "shopping_pt"])

# Merge the last shopping point features with our engineered history statistics
final_features_df = pd.merge(last_shopping_point, history_features, on="customer_ID")
print("Features DataFrame Shape:", final_features_df.shape)
final_features_df.head()

Features DataFrame Shape: (97009, 30)


,customer_ID,day,time,state,location,group_size,homeowner,car_age,car_value,risk_factor,...,F,G,cost,total_shopping_pts,cost_mean,cost_min,cost_max,cost_std,unique_cost_count,cost_change_amount
0,10000000,0,12:03,IN,10001,2,0,2,g,3.0,...,2,1,638,8,633.375000,630,638,3.961872,3,5
1,10000005,3,08:58,NY,10006,1,0,10,e,4.0,...,0,2,731,5,745.200000,730,755,13.423859,3,-24
2,10000007,4,08:43,PA,10008,1,0,11,c,NaN,...,0,1,605,7,609.285714,605,618,5.376315,4,-13
3,10000013,2,16:36,WV,10014,2,1,3,d,3.0,...,1,3,627,3,625.333333,620,629,4.725816,3,-2
4,10000014,4,16:43,MO,10015,1,0,5,d,3.0,...,2,2,603,5,599.600000,585,604,8.173127,3,-1


### Observation

We extracted the very last quote information requested by the customer right before they clicked "purchase". We then fused this state with our engineered historical metrics. This creates a powerful vector representing both the *current state* of the customer's request and their *historical journey*.

### Şərh / Müşahidə

Müştərinin "satın al" düyməsini sıxmazdan tam əvvəl baxdığı ən son təklif məlumatlarını çıxardıq. Daha sonra bu cari vəziyyəti yaratdığımız tarixi statistika göstəriciləri ilə birləşdirdik. Bu, həm müştərinin sorğusunun *cari vəziyyətini*, həm də onun *keçmiş addımlarını* təmsil edən güclü bir xüsusiyyətlər matrisi yaradır.

In [5]:
# Define target columns
target_cols = ["customer_ID", "A", "B", "C", "D", "E", "F", "G"]
targets_df = purchase_df[target_cols].copy()

# Rename target columns to distinguish them from feature columns
targets_df.columns = ["customer_ID"] + [f"target_{col}" for col in ["A", "B", "C", "D", "E", "F", "G"]]

# Merge features and targets securely on customer_ID
final_dataset = pd.merge(final_features_df, targets_df, on="customer_ID")
print("Final Dataset Shape (Features + Targets):", final_dataset.shape)

Final Dataset Shape (Features + Targets): (97009, 37)


### Observation

We extracted the true targets (the actual insurance package purchased) from `purchase_df` and merged them with our feature set using `customer_ID`. To avoid confusion during multi-output modeling, target choices are prefixed with `target_`. The dataset size is now exactly equal to the number of unique customers (97,009 rows).

### Şərh / Müşahidə

`purchase_df`-dən real hədəfləri (alınan real sığorta paketini) çıxardıq və onları `customer_ID` vasitəsilə xüsusiyyətlər çoxluğumuzla birləşdirdik. Çox-çıxışlı (multi-output) modelləşdirmə zamanı çaşqınlıq olmasın deyə, hədəf seçimlərinin əvvəlinə `target_` prefiksi əlavə etdik. Datasetin ölçüsü indi tam olaraq unikal müştərilərin sayına (97,009 sətir) bərabərdir.

In [6]:
# Handling Missing Values identified in EDA
# 1. risk_factor: Let's impute with median
risk_factor_median = final_dataset["risk_factor"].median()
final_dataset["risk_factor"] = final_dataset["risk_factor"].fillna(risk_factor_median)

# 2. C_previous & duration_previous: Impute with 0 or median (assuming no previous history means 0)
final_dataset["C_previous"] = final_dataset["C_previous"].fillna(0)
final_dataset["duration_previous"] = final_dataset["duration_previous"].fillna(0)

# 3. car_value: Missing values can be labeled as 'Unknown'
final_dataset["car_value"] = final_dataset["car_value"].fillna("Unknown")

# Verify missing values are resolved
print("Remaining missing values in features:")
print(final_dataset.drop(columns=[c for c in final_dataset.columns if 'target_' in c]).isnull().sum().sum())

Remaining missing values in features:
0


### Observation

Missing value imputation was carefully performed:
- `risk_factor` was filled using its median to preserve distribution balance.
- `C_previous` and `duration_previous` missing entries likely indicated a lack of previous insurance history, so they were imputed with `0`.
- Categorical missing values in `car_value` were designated as a distinct `'Unknown'` class to prevent loss of information.

### Şərh / Müşahidə

Çatışmayan (missing) dəyərlərin doldurulması diqqətlə yerinə yetirildi:
- `risk_factor` paylanma balansını qorumaq üçün median dəyəri ilə dolduruldu.
- `C_previous` və `duration_previous` boşluqları çox güman ki, əvvəlki sığorta tarixçəsinin olmadığını göstərirdi, buna görə də `0` ilə əvəzləndi.
- `car_value` sütunundakı kateqorial çatışmayan dəyərlər, hər hansı bir məlumat itkisinin qarşısını almaq üçün ayrıca `'Unknown'` (naməlum) sinfi olaraq təyin edildi.

In [7]:
# Encoding Categorical Features using Label Encoding
from sklearn.preprocessing import LabelEncoder

categorical_cols = ["state", "car_value"]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    final_dataset[col] = le.fit_transform(final_dataset[col].astype(str))
    label_encoders[col] = le

# Parsing time column to extract purely numeric hour info
final_dataset["hour"] = pd.to_datetime(final_dataset["time"], format="%H:%M").dt.hour
final_dataset = final_dataset.drop(columns=["time"])

final_dataset.head()

,customer_ID,day,state,location,group_size,homeowner,car_age,car_value,risk_factor,age_oldest,...,unique_cost_count,cost_change_amount,target_A,target_B,target_C,target_D,target_E,target_F,target_G,hour
0,10000000,0,10,10001,2,0,2,7,3.0,46,...,3,5,1,0,2,2,1,2,1,12
1,10000005,3,23,10006,1,0,10,5,4.0,28,...,3,-24,0,0,3,2,0,0,2,8
2,10000007,4,27,10008,1,0,11,3,3.0,43,...,4,-13,0,0,1,2,0,0,1,8
3,10000013,2,34,10014,2,1,3,4,3.0,62,...,3,-2,1,1,3,2,1,1,3,16
4,10000014,4,15,10015,1,0,5,4,3.0,32,...,3,-1,1,1,1,1,0,2,2,16


### Observation

Categorical variables (`state`, `car_value`) were transformed into numerical format using `LabelEncoder`, making them fully compatible with tree-based machine learning algorithms. Additionally, the string-based `time` feature was converted to a continuous numeric `hour` feature to extract cyclical day structures while dropping the redundant text format.

### Şərh / Müşahidə

Kateqorial dəyişənlər (`state`, `car_value`) `LabelEncoder` istifadə edilərək rəqəmsal formata salındı və ağac əsaslı maşın öyrənməsi alqoritmləri üçün tam uyğun hala gətirildi. Əlavə olaraq, mətn tipli `time` (saat) sütunu oxunaraq yalnız sırf rəqəmsal `hour` (saat) xüsusiyyətinə çevrildi və lazımsız mətn formatı silindi.

In [8]:
# Save the preprocessed and clean data to ready-to-use CSV files for Step 3 Modeling
features_only = final_dataset.drop(columns=[c for c in final_dataset.columns if 'target_' in c])
targets_only = final_dataset[["customer_ID"] + [f"target_{col}" for col in ["A", "B", "C", "D", "E", "F", "G"]]]

features_only.to_csv("preprocessed_features.csv", index=False)
targets_only.to_csv("preprocessed_targets.csv", index=False)

print("Preprocessed data successfully saved!")

Preprocessed data successfully saved!


### Observation

The preprocessing and feature engineering workflow is complete. The clean matrices have been exported to `preprocessed_features.csv` and `preprocessed_targets.csv`. The dataset is mathematically validated, contains zero missing entries, contains powerful history markers, and is structurally optimized for multi-output gradient boosting models.

### Şərh / Müşahidə

İlkin emal və xüsusiyyətlərin mühəndisliyi (feature engineering) iş axını tamamlandı. Təmizlənmiş matrislər `preprocessed_features.csv` və `preprocessed_targets.csv` fayllarına ixrac edildi. Dataset riyazi olaraq təsdiqləndi, daxilində heç bir boş xana qalmadı, güclü tarixçə göstəriciləri ilə zənginləşdirildi və çox-çıxışlı (multi-output) gradient boosting modelləri üçün struktur baxımından optimallaşdırıldı.